# 01 · Data preparation (run once, ~1 h on an RTX 4090 pod)

Everything the trainer needs is computed here and cached, so training reads nothing but in-RAM arrays.

| step | what | output |
|---|---|---|
| 1 | Hugging Face access to the gated Pocket TTS weights | HF cache |
| 2 | download Masri-100h (12.4 GB) | `raw/` |
| 3 | filter + normalize transcripts, verbalize numbers (Egyptian) | `work/manifest_*.jsonl` |
| 4 | CATT pass 1 → vote which words end in **ة** (the corpus writes ة as ه) | `work/catt_pass1.jsonl` |
| 5 | restore ة per word type, save the lexicon | `work/restored_*.jsonl`, `artifacts/ta_marbuta_lexicon.json` |
| 6 | CATT pass 2 on the restored text → diacritized variant of every word | `work/catt_pass2.jsonl` |
| 7 | character tokenizer (diacritics = separate tokens, ة+mark = one token) + unit tests | `artifacts/tokenizer.json` |
| 8 | CTC forced alignment, boundaries snapped to pauses, pause punctuation | `work/aligned_*.jsonl` |
| 9 | pocket-tts model configs (24-layer teacher, 6-layer student) | `artifacts/egy_*l.yaml` |
| 10 | Mimi latents + cold-start latents at every cut | `cache/{train,valid}/` |

Every step skips work that is already done, so re-running a cell is safe.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/workspace/EgyPocket-TTS")          # where you cloned the repo
os.environ.setdefault("EGY_DATA", "/workspace/egypocket_data")
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")     # models + HF token persist on the volume
DATA = Path(os.environ["EGY_DATA"])
CATT_DIR = Path("/workspace/catt_tashkeel")      # the uploaded CATT folder
os.chdir(REPO)
sys.path.insert(0, str(REPO))
PY = sys.executable                              # the egypocket conda env
print("python:", PY)
print("repo  :", REPO)
print("data  :", DATA)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
!df -h /workspace | tail -1

## 1. Hugging Face access

Training starts from Kyutai's **voice-cloning** Pocket TTS weights, which are gated:
1. open https://huggingface.co/kyutai/pocket-tts while logged in and accept the terms;
2. create a token at https://huggingface.co/settings/tokens (read access) and paste it below.

This cell then downloads the 24-layer teacher weights (~1.2 GB) and checks that the Mimi encoder is real (the ungated release ships an all-zero encoder, useless for training).

In [ ]:
from huggingface_hub import login
login()   # paste your token (or set HF_TOKEN in the environment and skip this line)

In [ ]:
import safetensors.torch
from pocket_tts.utils.config import CONFIGS_DIR, load_config
from pocket_tts.utils.utils import download_if_necessary

cfg = load_config(CONFIGS_DIR / "english_2026-04_24l.yaml")
weights = download_if_necessary(str(cfg.weights_path))
state = safetensors.torch.load_file(str(weights))
enc = max(v.abs().max().item() for k, v in state.items() if k.startswith("mimi.encoder."))
print(weights)
print("Mimi encoder max |w| =", enc, "-> OK" if enc > 0 else "-> ZERO ENCODER: accept the terms / check the token")
del state

## 2. Download the dataset (~12.4 GB, resumable)

In [ ]:
!{PY} -m egypocket.cli download

## 3. Manifests

Filters: no promo reads, ASR confidence ≥ 0.93, 1.5–20 s, plausible speaking rate. Digits in the transcripts (e.g. `سنه 1994`) are verbalized the Egyptian way (`الف وتسعمية واربعة وتسعين`).

In [ ]:
!{PY} -m egypocket.cli manifest --min-confidence 0.93

In [ ]:
import pandas as pd
from egypocket.data.jsonl import read_jsonl, read_json
display(pd.DataFrame(read_json(DATA / "work/manifest_stats.json")).T)
rows = read_jsonl(DATA / "work/manifest_train.jsonl")
for r in [r for r in rows if r["raw_text"] != r["text"]][:3]:
    print("raw :", r["raw_text"][:150]); print("norm:", r["text"][:150]); print()

## 4. CATT pass 1 and the `normalize_catt_output` rule

CATT emits `^ < >` (removed, never reach the training text) and `؞` (rewritten to `~`). Diacritics are **kept**. The demo below runs CATT on three sentences so you can see the raw and cleaned outputs.

In [ ]:
from egypocket.text.catt import CattDiacritizer, normalize_catt_output
catt_demo = CattDiacritizer(CATT_DIR, device="cpu")
demo = ["خدمة التقسيط في المدينة الكبيرة", "ونشوف مجموعة من اغرب الحاجات اللي كان بيعملها المصريين القدماء", "انا رحت المدينة امبارح وقابلت صاحبه"]
for raw in catt_demo.raw(demo):
    clean = normalize_catt_output(raw)
    assert not any(c in clean for c in "^<>\u061e")
    print("raw  :", raw); print("clean:", clean); print()
del catt_demo

In [ ]:
!{PY} -m egypocket.cli catt --pass 1 --catt-dir {CATT_DIR}

## 5. Restore ة

The transcripts write every ة as ه. For each word type ending in ه, CATT's vowel on the letter before it votes: fatha → ة (`حَاجَه`), damma → pronoun ه (`كِتَابُه`). Types with ≥ 60 % ة-votes are restored everywhere; the list is saved as a lexicon that the inference frontend applies too.

In [ ]:
!{PY} -m egypocket.cli restore

In [ ]:
rep = read_json(DATA / "work/ta_marbuta_report.json")
print(f"restored {rep['restored_types']} word types = {rep['restored_tokens']} tokens")
display(pd.DataFrame(rep["top_restored"]).head(30))
display(pd.DataFrame(rep["top_kept"]).head(30))

## 6. CATT pass 2 on the restored text (these diacritics are what the model learns from)

In [ ]:
!{PY} -m egypocket.cli catt --pass 2 --catt-dir {CATT_DIR}

In [ ]:
p2 = {r["id"]: r["diac"] for r in read_jsonl(DATA / "work/catt_pass2.jsonl")}
restored = read_jsonl(DATA / "work/restored_train.jsonl")
print("CATT failures:", sum(v is None for v in p2.values()), "of", len(p2))
for r in restored[:3]:
    print("plain:", r["text"][:120]); print("diac :", (p2[r["id"]] or "-")[:160]); print()

## 7. Tokenizer + unit tests

Characters, with every diacritic its own token and `ة`+mark merged (`ةْ ةَ ةِ ةُ ةً ةٌ ةٍ`), so `مَدِينَة` and `مَدِينَةْ` differ in exactly one, explicit token.

In [ ]:
!{PY} -m egypocket.cli tokenizer
!{PY} -m pytest -q tests

In [ ]:
from egypocket.text.tokenizer import load_tokenizer
from egypocket.text.frontend import normalize_text
tok = load_tokenizer(DATA / "artifacts/tokenizer.json")
print("vocab size:", tok.get_vocab_size())
for s in ["مَدِينَة", "مَدِينَةْ", "مَدِينَةَ", "مَدِينَةِ", "مَدِينَةُ", "ق؞َابلت"]:
    n = normalize_text(s)
    print(f"{s:>12} -> {[tok.id_to_token(i) for i in tok.encode(n).ids]}")

## 8. Forced alignment (~15–25 min)

wav2vec2 CTC (Arabic) word timings, boundaries moved into measured pauses, pause punctuation (`,` ≥ 0.22 s, `.` ≥ 0.55 s), CTC score per clip.

In [ ]:
!{PY} -m egypocket.cli align --batch-size 16

In [ ]:
import numpy as np, matplotlib.pyplot as plt
al = read_jsonl(DATA / "work/aligned_train.jsonl")
scores = np.array([r["ctc_score"] for r in al])
cut = np.quantile(scores, 0.03)
plt.figure(figsize=(8, 3)); plt.hist(scores, bins=80); plt.axvline(cut, color="r"); plt.title("CTC score per clip (red = default cut-off: worst 3%)"); plt.show()
print("clips:", len(al), f"| cut-off {cut:.2f}")
for r in sorted(al, key=lambda r: r["ctc_score"])[:3]:
    print(f"worst {r['ctc_score']:.2f}:", r["text"][:120])
from IPython.display import Audio
r = al[0]
print(r["text"]); print([(w["word"], w["start"], w["end"]) for w in r["words"][:8]])
Audio(r["audio"])

## 9. Model configs

In [ ]:
!{PY} -m egypocket.cli configs

## 10. Latent cache (~10 min)

The worst 3 % of clips by CTC score (transcript and audio disagree most) are left out (`--drop-worst 0.03`; use `--min-ctc-score X` for an absolute cut-off after looking at the histogram above).

In [ ]:
!{PY} -m egypocket.cli cache --drop-worst 0.03

In [ ]:
summary = read_json(DATA / "cache/summary.json")
print(json.dumps(summary, indent=1))

### Listen to what the trainer sees (decoded from the cache) and look at the text variants

In [ ]:
import torch, random
from IPython.display import Audio, display
from egypocket.data.loader import LatentCache, CachedBatches
from egypocket.data.cache import load_mimi
from egypocket.training.args import DiacriticArgs
from egypocket.training.train import decode_latents

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mimi = load_mimi(DATA / "artifacts/egy_24l.yaml", dev)
cache = LatentCache(DATA / "cache/train")
loader = CachedBatches(cache, lambda t: tok.encode(t).ids, 4, seed=0, diacritics=DiacriticArgs(), prompt_trim_max_sec=0.5, final_punct_dropout=0.3)
for i in random.Random(0).sample(range(len(cache)), 3):
    tokens, text, prompt, target = loader.sample(i)
    print(text)
    display(Audio(decode_latents(mimi, torch.from_numpy(prompt).to(dev)), rate=24000))
    display(Audio(decode_latents(mimi, torch.from_numpy(target).to(dev)), rate=24000))
del mimi; torch.cuda.empty_cache()

Data is ready → open **02_train_teacher.ipynb**.